In [1]:
# 구글 드라이브 연동
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
# !unzip /content/VL.zip -d /content/VL/
!unzip "/content/drive/MyDrive/AI_hub_71557/138.뉴스 대본 및 앵커 음성 데이터/01-1.정식개방데이터/Training/02.라벨링데이터/TL.zip" -d /content/TL/

스트리밍 출력 내용이 길어서 마지막 5000줄이 삭제되었습니다.
   creating: /content/TL/SPK088/SPK088YTNLO633/
  inflating: /content/TL/SPK088/SPK088YTNLO633/SPK088YTNLO633M005.json  
  inflating: /content/TL/SPK088/SPK088YTNLO633/SPK088YTNLO633M003.json  
  inflating: /content/TL/SPK088/SPK088YTNLO633/SPK088YTNLO633M004.json  
  inflating: /content/TL/SPK088/SPK088YTNLO633/SPK088YTNLO633M001.json  
  inflating: /content/TL/SPK088/SPK088YTNLO633/SPK088YTNLO633M002.json  
   creating: /content/TL/SPK088/SPK088YTNLO634/
  inflating: /content/TL/SPK088/SPK088YTNLO634/SPK088YTNLO634M004.json  
  inflating: /content/TL/SPK088/SPK088YTNLO634/SPK088YTNLO634M002.json  
  inflating: /content/TL/SPK088/SPK088YTNLO634/SPK088YTNLO634M001.json  
  inflating: /content/TL/SPK088/SPK088YTNLO634/SPK088YTNLO634M003.json  
  inflating: /content/TL/SPK088/SPK088YTNLO634/SPK088YTNLO634M005.json  
  inflating: /content/TL/SPK088/SPK088YTNLO634/SPK088YTNLO634M006.json  
   creating: /content/TL/SPK088/SPK088YTNLO635/
  inflating: /con

In [3]:
# !unzip /content/TL.zip -d /content/TL/
!unzip "/content/drive/MyDrive/AI_hub_71557/138.뉴스 대본 및 앵커 음성 데이터/01-1.정식개방데이터/Validation/02.라벨링데이터/VL.zip" -d /content/VL/

스트리밍 출력 내용이 길어서 마지막 5000줄이 삭제되었습니다.
  inflating: /content/VL/SPK076/SPK076YTNPO949/SPK076YTNPO949F003.json  
  inflating: /content/VL/SPK076/SPK076YTNPO949/SPK076YTNPO949F001.json  
  inflating: /content/VL/SPK076/SPK076YTNPO949/SPK076YTNPO949F004.json  
   creating: /content/VL/SPK076/SPK076YTNPO950/
  inflating: /content/VL/SPK076/SPK076YTNPO950/SPK076YTNPO950F004.json  
  inflating: /content/VL/SPK076/SPK076YTNPO950/SPK076YTNPO950F005.json  
  inflating: /content/VL/SPK076/SPK076YTNPO950/SPK076YTNPO950F001.json  
  inflating: /content/VL/SPK076/SPK076YTNPO950/SPK076YTNPO950F003.json  
  inflating: /content/VL/SPK076/SPK076YTNPO950/SPK076YTNPO950F002.json  
   creating: /content/VL/SPK076/SPK076YTNPO951/
  inflating: /content/VL/SPK076/SPK076YTNPO951/SPK076YTNPO951F003.json  
  inflating: /content/VL/SPK076/SPK076YTNPO951/SPK076YTNPO951F005.json  
  inflating: /content/VL/SPK076/SPK076YTNPO951/SPK076YTNPO951F002.json  
  inflating: /content/VL/SPK076/SPK076YTNPO951/SPK076YTNPO951F004

In [4]:
import json
import os
import re

In [6]:
# json 형식 확인
json_path = "/content/VL/SPK003/SPK003YTNSO162/SPK003YTNSO162F001.json"

with open(json_path, "r", encoding="utf-8") as f:
    data = json.load(f)

print(json.dumps(data, indent=4, ensure_ascii=False))  # json 출력


{
    "script": {
        "id": "YTNSO162",
        "url": "http://www.ytn.co.kr/_ln/0103_201801110324227695",
        "title": "맥도날드 납품 업체 임원 또 구속 피해",
        "press": "YTN",
        "press_field": "사회",
        "press_date": "20180111",
        "index": 1,
        "text": "이른바 햄버거병 유발 가능성이 있는 햄버거용 패티를 맥도날드에 납품한 혐의로 영장이 청구된 업체 임원들이 또 구속 위기를 피했습니다.",
        "sentence_type": "작문형",
        "keyword": "패티,맥도날드,구속 피해,맥도날드 햄버거,이른바,구속 영장,이른바 햄버거,축산물,소고기 패티,햄버거용 패티"
    },
    "speaker": {
        "id": "SPK003",
        "age": "40대",
        "sex": "여성",
        "job": "전직아나운서"
    },
    "file_information": {
        "audio_format": "44100 Hz 16bit PCM",
        "utterance_start": "0.493",
        "utterance_end": "12.424",
        "audio_duration": "12.829"
    }
}


In [7]:
def extract_transcriptions(root_folder: str, output_file_path: str, mode='w'):
    """
    주어진 루트 폴더에서 JSON 파일을 찾아 오디오 파일 이름과 전사 텍스트를 추출하여 저장하는 함수.

    Parameters:
        root_folder (str): JSON 파일들이 포함된 최상위 폴더 경로
        output_file_path (str): 추출된 텍스트를 저장할 txt 파일 경로
    """
    try:
        with open(output_file_path, mode, encoding='utf-8') as output_file:
            # 루트 폴더 내 하위 폴더 탐색
            for spk_folder in os.listdir(root_folder):
                spk_folder_path = os.path.join(root_folder, spk_folder)

                if os.path.isdir(spk_folder_path):  # 상위 폴더인지 확인
                    for sub_folder in os.listdir(spk_folder_path):  # 하위 폴더 순회
                        sub_folder_path = os.path.join(spk_folder_path, sub_folder)

                        if os.path.isdir(sub_folder_path):  # 폴더인지 확인
                            for file_name in os.listdir(sub_folder_path):  # JSON 파일 찾기
                                if file_name.endswith('.json'):
                                    json_file_path = os.path.join(sub_folder_path, file_name)

                                    try:
                                        # JSON 파일 읽기
                                        with open(json_file_path, 'r', encoding='utf-8') as json_file:
                                            data = json.load(json_file)

                                            # 파일 이름에서 확장자 제거하여 audio_name 추출
                                            audio_name = file_name.rsplit('.', 1)[0]

                                            # 전사 텍스트 추출
                                            transcription = data.get('script', {}).get('text', '')

                                            if transcription:  # 텍스트가 존재하는 경우만 기록
                                                output_file.write(f"{audio_name}:{transcription}\n")
                                            else:
                                                print(f"전사 텍스트 없음: {json_file_path}")

                                    except Exception as e:
                                        print(f"!오류 발생: {json_file_path}, 오류 메시지: {e}")

        print(f"텍스트 파일이 성공적으로 저장되었습니다: {output_file_path}")

    except Exception as e:
        print(f"!파일 저장 중 오류 발생: {e}")

In [9]:
def check_transcriptions():
  """
  txt 파일의 정보를 확인하는 함수.

  """
  file_path = "/content/text_only.txt"

  with open(file_path, "r", encoding="utf-8") as file:
      lines = file.readlines()

  # 총 줄 수 출력
  print(f"총 줄 수: {len(lines)}")

  # 상위 5줄 출력
  print("상위 5줄")
  for line in lines[:5]:
      print(line.strip())  # strip()은 줄바꿈 문자를 제거

  print("---------------------------------------------")

  print("하위 5줄")
  # 하위 5줄 출력
  for line in lines[-5:]:
      print(line.strip())  # strip()은 줄바꿈 문자를 제거

In [10]:
# 출력할 텍스트 파일 경로
output_file_path = '/content/text_only.txt'

In [11]:
extract_transcriptions("/content/TL", output_file_path)

텍스트 파일이 성공적으로 저장되었습니다: /content/text_only.txt


In [12]:
check_transcriptions()

총 줄 수: 306734
상위 5줄
SPK080YTNIN974M002:글로벌 영화시장 분석 기관 가워 스트리트는 현지시간 (26)/(이십 육) 일 이러한 내용의 보고서를 발표했다고, 미국 영화 전문 매체 할리우드 리포터 등이 보도했습니다.
SPK080YTNIN974M005:가워 스트리트는 올해 하반기 중국에서 한국전쟁을 소재로 한 영화 `장진호`가 흥행에 성공한데다, (007)/(공 공 칠) 시리즈 `노 타임 투 다이` 등 할리우드 블록버스터 영화가 다시 인기몰이에 나섰다는 점을 고려해 내년 박스오피스 전망치를 올렸다고 설명했습니다.
SPK080YTNIN974M003:가워 스트리트는 올해 전 세계 극장가가 코로나 (19)/(일 구) 로 타격을 받았지만, 내년에는 (216)/(이백 십 육) 억 달러, 우리돈 (25)/(이십 오) 조 (2)/(이) 천억 원 규모의 티켓 매출을 기록할 것으로 예상했습니다.
SPK080YTNIN974M008:할리우드 영화 `분노의 질주, 더 얼티메이트`는 (7)/(칠) 억 (2)/(이) 천 (100)/(백) 만 달러 티켓 판매를 올려 (3)/(삼) 위를 차지했습니다.
SPK080YTNIN974M006:가워 스트리트는 중국의 글로벌 박스오피스 점유율이 올해 (28%)/(이십 팔 퍼센트) 에서 내년에는 (34%)/(삼십 사 퍼센트) 로 상승할 것으로 전망했습니다.
---------------------------------------------
하위 5줄
SPK073KBSCU096M005:그저 피아노 선율이 이끄는 대로 몸을 맡기고 들어와 눈과 귀, 마음을 열어주기만 하면 됩니다.
SPK073KBSCU096M004:가던 걸음을 멈추고 잠시 귀를 기울이는 시민들도 있고, 활짝 열린 문으로 아예 들어와서 자리를 잡고 앉아 경청하기도 합니다.
SPK073KBSCU096M003:쉬너스 씨의 음악 연주는 매일 (5)/(다섯) 시간씩 계속됩니다.
SPK073KBSCU096M002:음악의 아버지 `바흐`의 곡입니다.
SPK073KBSCU096M007

In [13]:
extract_transcriptions("/content/VL", output_file_path, 'a'); # 'a'는 이어쓰기

텍스트 파일이 성공적으로 저장되었습니다: /content/text_only.txt


In [14]:
check_transcriptions() # 추가 된 것을 확인 text_only.txt = TL + VL

총 줄 수: 341860
상위 5줄
SPK080YTNIN974M002:글로벌 영화시장 분석 기관 가워 스트리트는 현지시간 (26)/(이십 육) 일 이러한 내용의 보고서를 발표했다고, 미국 영화 전문 매체 할리우드 리포터 등이 보도했습니다.
SPK080YTNIN974M005:가워 스트리트는 올해 하반기 중국에서 한국전쟁을 소재로 한 영화 `장진호`가 흥행에 성공한데다, (007)/(공 공 칠) 시리즈 `노 타임 투 다이` 등 할리우드 블록버스터 영화가 다시 인기몰이에 나섰다는 점을 고려해 내년 박스오피스 전망치를 올렸다고 설명했습니다.
SPK080YTNIN974M003:가워 스트리트는 올해 전 세계 극장가가 코로나 (19)/(일 구) 로 타격을 받았지만, 내년에는 (216)/(이백 십 육) 억 달러, 우리돈 (25)/(이십 오) 조 (2)/(이) 천억 원 규모의 티켓 매출을 기록할 것으로 예상했습니다.
SPK080YTNIN974M008:할리우드 영화 `분노의 질주, 더 얼티메이트`는 (7)/(칠) 억 (2)/(이) 천 (100)/(백) 만 달러 티켓 판매를 올려 (3)/(삼) 위를 차지했습니다.
SPK080YTNIN974M006:가워 스트리트는 중국의 글로벌 박스오피스 점유율이 올해 (28%)/(이십 팔 퍼센트) 에서 내년에는 (34%)/(삼십 사 퍼센트) 로 상승할 것으로 전망했습니다.
---------------------------------------------
하위 5줄
SPK073KBSPO059M008:아울러 월스트리트 저널은 북한군내 부패와 이로 인한 처형이 증가하고 있다는 브룩스 한미연합사령관의 말을 인용하면서, 이 역시 훈련 축소의 원인이 될 수도 있다고 시사했습니다.
SPK073KBSPO059M005:북한군의 동계 훈련 축소는 무엇보다 대북 제재의 여파인 것으로 분석됐습니다.
SPK073KBSPO059M003:그러나 이번 겨울에는 김정은이 동계 군사 훈련을 참관하는 일이 거의 없어졌습니다.
SPK073KBSPO059M006:북한전문매체 (38)

In [47]:
def preprocess_text(input_file_path, output_file_path):
  """
  txt 파일의 전사 텍스트를 전처리 하여 저장하는 함수.

  Parameters:
      input_file_path (str): txt 폴더 경로
      output_file_path (str): 전처리 된 텍스트를 저장할 txt 파일 경로
  """
  # 예외 패턴 1: `(앞)(뒤)` 패턴 (오탈자: 슬래시 누락)
  # SPK080YTNLO561M006:(1,600)/(천 육백) 명 정도가 전국에 있고, (75%)(칠십 오 퍼센트) 가 수도권에 있군요.
  # (75%)(칠십 오 퍼센트) => 중간에 / 생략
  pattern_missed_slash = re.compile(r'\(([^()]+)\)\(([^()]+)\)')

  # 예외 패턴 2: `(앞)/(뒤) 단어)` → `(앞)/(뒤 단어)`**
  # SPK069YTNEC780F002:현지시간 지난달 (31)/(삼십 일) 일 뉴욕상업거래소에서 (5)/(오) 월 인도분 서부 텍사스산 원유는 (1)/(일) 배럴에 (7)/(칠) 달러 (54)/(오십 사) 센트, (7%)/(칠) 퍼센트) 내린 (100)/(백) 달러 (28)/(이십 팔) 센트에서 거래를 마쳤습니다.
  # (7%)/(칠) 퍼센트) -> (7%)/(칠 퍼센트)로 변경 후 처리
  pattern_fix_closing_bracket = re.compile(r'\(([^()]+)\)/\(([^()]+)\) ([^()]+)\)')

  # 예외 패턴 3: `(앞)/(앞)/(뒤)` 패턴 (중복 앞 처리)
  # SPK089YTNLO595M001:사망사고가 잇달아 발생한 현대건설의 주요 시공 현장에 대해 정부가 감독을 실시한 결과 모두 (254)/(254)/(이백 쉰 네) 건의 산업안전보건법 위반사항이 적발됐습니다.
  # (254)/(254)/(이백 쉰 네) -> (254)/(이백 쉰 네)로 변경 후 처리
  pattern_duplicate = re.compile(r'\(([^()]+)\)/\(\1\)/\(([^()]+)\)')

  # 일반 정규화 패턴 : `(앞)/(뒤)` 패턴 (기본 변환)
  pattern_correct = re.compile(r'\(([^()]+)\)/\(([^()]+)\)')

  # 문장의 개수 카운팅
  processing_count = 0
  no_processing_count = 0
  error_count = 0
  fixed_bracket_count = 0

  # 파일 읽기 및 전처리 수행
  with open(input_file_path, 'r', encoding='utf-8') as infile, \
      open(output_file_path, 'w', encoding='utf-8') as outfile:

      for line in infile:
          line = line.strip()  # 공백 제거
          if not line:
              continue  # 빈 줄 건너뛰기

          # "audio_name: 원본 텍스트" 형식에서 분리
          if ":" not in line:
              continue  # 잘못된 형식 무시
          audio_name, text = line.split(":", 1)

          # 1 `(앞)(뒤)` 패턴 처리 (오탈자 수정)
          text = pattern_missed_slash.sub(r'(\1)/(\2)', text)

          # 2 `(앞)/(뒤) 단어)` → `(앞)/(뒤 단어)` 수정
          if pattern_fix_closing_bracket.search(text):
              text = pattern_fix_closing_bracket.sub(r'(\g<1>)/(\g<2> \g<3>)', text)

          # 3 `(앞)/(앞)/(뒤)` 패턴 처리 (중복 앞 제거)
          # processed_text_norm1 = pattern_duplicate.sub(r'\1', text)
          # processed_text_norm2 = pattern_duplicate.sub(r'\2', text)
          text = pattern_duplicate.sub(r'(\1)/(\2)', text)

          # 4 `(앞)/(뒤)` 패턴 처리 (일반 변환)
          processed_text_norm1 = pattern_correct.sub(r'\1', text)
          processed_text_norm2 = pattern_correct.sub(r'\2', text)

          # 디버깅: 제대로 처리되지 않은 경우 카운트
          if processed_text_norm1 == text and processed_text_norm2 == text:
              no_processing_count += 1
              outfile.write(f"{audio_name}:{text} $전처리할 필요가 없는 문장이다.\n")
          else:
              processing_count += 1
              if processed_text_norm1 == processed_text_norm2:  # 동일한 결과일 때 오류 체크
                  error_count += 1
                  print(f"!오류 발생: {audio_name}에서 전처리가 제대로 이루어지지 않았습니다.")
              outfile.write(f"{audio_name}:{processed_text_norm1} ${processed_text_norm2}\n")

  # 최종 디버깅 출력
  print(f"총 문장: {processing_count+no_processing_count}개")
  print(f"전처리 된 문장: {processing_count}개")
  print(f"전처리할 필요 없는 문장: {no_processing_count}개")
  if error_count > 0:
      print(f"!오류 발생한 문장: {error_count}개")
  else:
      print("모든 문장이 정상 처리되었습니다!")

In [48]:

# 원본 및 출력 파일 경로
input_file_path = "/content/text_only.txt"
output_file_path = "/content/text_preprocessed.txt"

preprocess_text(input_file_path, output_file_path)

총 문장: 341860개
전처리 된 문장: 180344개
전처리할 필요 없는 문장: 161516개
모든 문장이 정상 처리되었습니다!


In [50]:
# 예외 처리 확인
with open(input_file_path, "r", encoding="utf-8") as file:
    lines = file.readlines()

print("전처리 전 텍스트")
# 이상치 확인
for line in lines:
    if 'SPK089YTNLO595M001' in line:
        print(line.strip())

print()
print("전처리 후 텍스트")
with open(output_file_path, "r", encoding="utf-8") as file:
    lines = file.readlines()

# 이상치 확인
for line in lines:
    if 'SPK089YTNLO595M001' in line:
        print(line.strip())


전처리 전 텍스트
SPK089YTNLO595M001:사망사고가 잇달아 발생한 현대건설의 주요 시공 현장에 대해 정부가 감독을 실시한 결과 모두 (254)/(254)/(이백 쉰 네) 건의 산업안전보건법 위반사항이 적발됐습니다.

전처리 후 텍스트
SPK089YTNLO595M001:사망사고가 잇달아 발생한 현대건설의 주요 시공 현장에 대해 정부가 감독을 실시한 결과 모두 254 건의 산업안전보건법 위반사항이 적발됐습니다. $사망사고가 잇달아 발생한 현대건설의 주요 시공 현장에 대해 정부가 감독을 실시한 결과 모두 이백 쉰 네 건의 산업안전보건법 위반사항이 적발됐습니다.
